In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
from pyspark.sql.functions import col

In [ ]:
dbutils.widgets.dropdown("target", "all", [
    "all", "fact_calidad_aire", "fact_trafico", "fact_trafico_diario",
])

TARGET = dbutils.widgets.get("target")
NOTEBOOK = "gold/facts"

errors = []
built = []

In [ ]:
# Grain: estacion x fecha x magnitud
if TARGET in ("fact_calidad_aire", "all"):
    try:
        silver_aire = spark.table(f"{SILVER_TABLE}.aire")
        dim_estacion = spark.table(f"{GOLD_TABLE}.dim_estacion_aire")

        fact_aire = (
            silver_aire.alias("a")
            .join(
                dim_estacion.alias("e"),
                col("a.estacion").cast("int") == col("e.codigo_corto").cast("int"),
            )
            .select(
                col("a.estacion").cast("int").alias("estacion"),
                col("e.cod_dis").alias("cod_dis"),
                col("a.magnitud"),
                col("a.fecha"),
                col("a.dato"),
                col("a.validez"),
            )
        )

        rows = fact_aire.count()
        if rows == 0:
            raise Exception("fact_calidad_aire join produced 0 rows")
        if not write_gold(fact_aire, "fact_calidad_aire"):
            raise Exception("write_gold returned False")
        built.append("fact_calidad_aire")
        print(f"fact_calidad_aire: {rows} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail fact_calidad_aire: {type(e).__name__}: {e}")

In [ ]:
# Grain: punto x timestamp
if TARGET in ("fact_trafico", "all"):
    try:
        silver_trafico = spark.table(f"{SILVER_TABLE}.trafico")
        dim_punto = spark.table(f"{GOLD_TABLE}.dim_punto_trafico")

        fact_trafico = (
            silver_trafico.alias("t")
            .join(dim_punto.alias("p"), col("t.id").cast("int") == col("p.id").cast("int"))
            .select(
                col("t.id").cast("int").alias("id"),
                col("p.distrito"),
                col("t.fecha"),
                col("t.intensidad"),
                col("t.ocupacion"),
                col("t.carga"),
                col("t.vmed"),
                col("t.error"),
            )
        )

        rows = fact_trafico.count()
        if rows == 0:
            raise Exception("fact_trafico join produced 0 rows")
        if not write_gold(fact_trafico, "fact_trafico"):
            raise Exception("write_gold returned False")
        built.append("fact_trafico")
        print(f"fact_trafico: {rows} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail fact_trafico: {type(e).__name__}: {e}")

In [ ]:
# Daily rollup for coverage
if TARGET in ("fact_trafico_diario", "all"):
    try:
        fact_diario = spark.sql(f"""
            SELECT t.id,
                   p.distrito,
                   CAST(t.fecha AS DATE) AS fecha,
                   avg(t.intensidad) AS intensidad_media,
                   avg(t.ocupacion) AS ocupacion_media,
                   avg(t.carga) AS carga_media,
                   avg(t.vmed) AS vmed_media,
                   count(*) AS lecturas,
                   sum(CASE WHEN t.error = 'N' THEN 1 ELSE 0 END) AS lecturas_ok
            FROM {SILVER_TABLE}.trafico t
            JOIN {GOLD_TABLE}.dim_punto_trafico p
              ON CAST(t.id AS INT) = CAST(p.id AS INT)
            GROUP BY t.id, p.distrito, CAST(t.fecha AS DATE)
        """)

        rows = fact_diario.count()
        if rows == 0:
            raise Exception("fact_trafico_diario produced 0 rows")
        if not write_gold(fact_diario, "fact_trafico_diario"):
            raise Exception("write_gold returned False")
        built.append("fact_trafico_diario")
        print(f"fact_trafico_diario: {rows} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail fact_trafico_diario: {type(e).__name__}: {e}")

In [ ]:
print(f"facts (target={TARGET}) built={built} failed={len(errors)}")
log_errors(errors)